# ML Zoomcamp 2026 - Homework 6
Decision Trees and Ensemble Learning (pinned 2026 car fuel-efficiency release)

In [1]:
!pip install xgboost  # if needed

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## Preparing the dataset

Fill missing values with zeros, split with `random_state=1`, encode with `DictVectorizer(sparse=True)`.

In [3]:
!wget -q -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.shape                      # (10000, 11)
df.isnull().sum()             # horsepower 877, acceleration 264, rest 0

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [4]:
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

X_train.shape, X_val.shape   # (6000, 16) (2000, 16)

((6000, 16), (2000, 16))

In [5]:
def rmse(y, y_pred):
    return np.sqrt(mean_squared_error(y, y_pred))

## Q1. Decision tree with `max_depth=1`

In [6]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

# |--- model_year <= 1997.50
# |   |--- value: [28.61]
# |--- model_year >  1997.50
# |   |--- value: [31.17]

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [7]:
# The single root split is on model_year  ->  answer: model_year

## Q2. Random forest, `n_estimators=10`

In [8]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

score = rmse(y_val, rf.predict(X_val))
score, round(score, 3)   # 1.837054 -> 1.837

(np.float64(1.837054054730018), np.float64(1.837))

## Q3. Tuning `n_estimators`

In [9]:
scores = {}
for n in [10, 50, 100, 150]:
    m = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    m.fit(X_train, y_train)
    s = rmse(y_val, m.predict(X_val))
    scores[n] = s
    print(f'n_estimators={n:<5} rmse={s:.6f} -> {round(s, 3)}')

# n_estimators=10    rmse=1.837054 -> 1.837
# n_estimators=50    rmse=1.771567 -> 1.772
# n_estimators=100   rmse=1.768136 -> 1.768   <-- lowest
# n_estimators=150   rmse=1.768804 -> 1.769

min(scores, key=scores.get)   # 100

n_estimators=10    rmse=1.837054 -> 1.837
n_estimators=50    rmse=1.771567 -> 1.772
n_estimators=100   rmse=1.768136 -> 1.768
n_estimators=150   rmse=1.768804 -> 1.769


100

In [10]:
# NOTE: the 100 vs 150 margin is only 0.0007 - one unit in the 3rd decimal.
# Re-running under scikit-learn 1.7.2 gives byte-identical numbers, so this is stable,
# but it is the tightest call in the homework.

## Q4. Tuning `max_depth` (mean RMSE over `n_estimators`)

In [11]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        m = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        m.fit(X_train, y_train)
        scores.append(rmse(y_val, m.predict(X_val)))
    print(f'max_depth={d:<4} mean_rmse={np.mean(scores):.6f} -> {round(np.mean(scores), 3)}'
          f'   per-n={[round(s, 3) for s in scores]}')

# max_depth=10  mean_rmse=1.755405 -> 1.755   <-- lowest
# max_depth=15  mean_rmse=1.783953 -> 1.784
# max_depth=20  mean_rmse=1.788242 -> 1.788
# max_depth=25  mean_rmse=1.786317 -> 1.786

max_depth=10   mean_rmse=1.755405 -> 1.755   per-n=[np.float64(1.784), np.float64(1.749), np.float64(1.745), np.float64(1.745)]
max_depth=15   mean_rmse=1.783953 -> 1.784   per-n=[np.float64(1.83), np.float64(1.771), np.float64(1.768), np.float64(1.767)]
max_depth=20   mean_rmse=1.788242 -> 1.788   per-n=[np.float64(1.841), np.float64(1.773), np.float64(1.769), np.float64(1.77)]
max_depth=25   mean_rmse=1.786317 -> 1.786   per-n=[np.float64(1.837), np.float64(1.772), np.float64(1.768), np.float64(1.768)]


## Q5. Feature importance

In [12]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False)

importance.head(10)

,feature,importance
9,model_year,0.296001
15,vehicle_weight,0.203879
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201
14,origin=USA,0.042144
2,drivetrain=Front-wheel drive,0.020145
11,num_doors,0.019415
5,fuel_type=Diesel,0.017038


In [13]:
# overall top feature is model_year (0.296), but it is NOT one of the 4 options.
# Among the listed candidates:
importance[importance.feature.isin(
    ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement'])]

# vehicle_weight        0.203879   <-- highest of the four
# horsepower            0.078017
# engine_displacement   0.061005
# acceleration          0.055201

,feature,importance
15,vehicle_weight,0.203879
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201


## Q6. XGBoost - tuning `eta`

In [14]:
features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

In [15]:
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=25)

    print(f'eta={eta}:  train_rmse={rmse(y_train, model.predict(dtrain)):.6f}'
          f'   val_rmse={rmse(y_val, model.predict(dval)):.6f}')
    print()

# eta=0.3:  train_rmse=0.901056   val_rmse=1.826458
# eta=0.1:  train_rmse=1.311918   val_rmse=1.724810   <-- best on validation

[0]	train-rmse:2.40653	val-rmse:2.46782
[25]	train-rmse:1.35837	val-rmse:1.74359
[50]	train-rmse:1.18645	val-rmse:1.77517
[75]	train-rmse:1.02727	val-rmse:1.80682
[99]	train-rmse:0.90106	val-rmse:1.82646
eta=0.3:  train_rmse=0.901056   val_rmse=1.826458

[0]	train-rmse:2.74739	val-rmse:2.78635
[25]	train-rmse:1.50918	val-rmse:1.72751
[50]	train-rmse:1.40368	val-rmse:1.71358
[75]	train-rmse:1.35099	val-rmse:1.72057
[99]	train-rmse:1.31192	val-rmse:1.72481
eta=0.1:  train_rmse=1.311918   val_rmse=1.724810



## Answers
| Q | Question | Answer |
|---|---|---|
| 1 | Feature used for the first split | `model_year` |
| 2 | RMSE of random forest (10 trees) | 1.837 |
| 3 | Best `n_estimators` | 100 (RMSE 1.768) |
| 4 | Best `max_depth` | 10 (mean RMSE 1.755) |
| 5 | Most important feature (of the four) | `vehicle_weight` (0.2039) |
| 6 | Best `eta` | 0.1 (val RMSE 1.725 vs 1.826) |

### Verification

All six answers were reproduced byte-for-byte under two environments:

| | scikit-learn | xgboost | numpy |
|---|---|---|---|
| run 1 | 1.6.1 | 3.4.1 | 2.3.5 |
| run 2 | 1.7.2 | 3.2.0 | 2.3.3 |

### Two things worth noticing

**Q4 - limiting depth beats unlimited depth.** `max_depth=10` has the *best* mean RMSE (1.755),
while the deeper forests (20, 25) are barely better than the untuned baseline from Q3 (1.768).
Growing trees all the way down lets the forest memorise noise; capping depth is a regulariser,
not just a speed knob.

**Q6 - the slower learning rate wins, and it's not close.** `eta=0.1` beats `eta=0.3` by 0.10 RMSE.
Look at the training errors: 0.901 for `eta=0.3` against 1.312 for `eta=0.1`. The aggressive model
fits the training set far better and generalises worse - textbook overfitting. Lower `eta` with a
fixed round budget acts as regularisation, the same role `max_depth` plays in Q4.

**Watch out for Q5.** The single most important feature overall is `model_year` (0.296), which is
not among the four options. The question restricts you to a subset, and within that subset
`vehicle_weight` wins at 0.204 - nearly 3x the next candidate.